# Voxtral-4B-TTS — candidat voix de la brique A

Synthétise **les dix mêmes phrases** que les candidats Qwen, sur **la même voix SIWIS**,
pour que la comparaison isole le moteur et non le timbre.

Chiffres à battre (mesurés le 27/08, `data/bakeoff/` en local) :

| candidat | UTMOS | WER aller-retour | verdict à l'oreille |
|---|---|---|---|
| `qwen_siwis_neut` | **3,99** | 0,000 | « mieux, mais on entend le robot » |
| `qwen_preset` | 3,57 | 0,000 | accent anglais |
| incumbent `dialogue-tts-1000h` | 3,73 | — | — |
| anglais du modèle lui-même | 4,08 | — | plafond de référence |

## Les cinq pièges déjà payés (tous encodés ci-dessous)

1. **Pas de poids transformers** — Voxtral-TTS ne se charge qu'avec vLLM-Omni.
2. **`vllm-omni` ne déclare pas `vllm`** et les versions sont appariées → `vllm-omni==0.22.0`
   avec le wheel `0.22.1+cu129`. « La dernière de chaque » casse sur une API manquante.
3. **Le wheel est buildé CUDA 13** → précharger `nvidia/cu13/lib` (et **pas** le premier
   `nvidia/*/lib` venu : mon garde-fou a chargé `cublas` en croyant charger cu13).
4. **L'installation vLLM casse `torchaudio`** (`undefined symbol: torch_library_impl`) →
   résoudre la référence SIWIS **AVANT** d'installer (cellule 3 avant cellule 4).
5. **Dernier échec en date, non résolu** : `from vllm import SamplingParams` lève un
   `AssertionError` nu depuis `vllm/__init__.py:70`. C'est là qu'il faut chercher.

In [ ]:
# 1. GPU + repo. Voxtral-4B tient sur >=16 Go.
import subprocess, sys, os
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'PAS DE GPU')

REPO = '/content/finetuning_s2s_toolcalling'
BRANCH = 'rd/pr_rca_eval_baseline'
if not os.path.exists(REPO + '/pyproject.toml'):
    !git clone -q --branch {BRANCH} https://github.com/rcarvalo/finetuning_s2s_toolcalling.git {REPO}
!cd {REPO} && git fetch -q origin {BRANCH} && git reset -q --hard origin/{BRANCH} && git log --oneline -1

## 2. La référence SIWIS — AVANT toute installation

Le résolveur lit et transcrit le clip avec `torchaudio`, que l'installation vLLM cassera
à la cellule suivante. On la résout donc maintenant et on garde le résultat sur disque.

Le registre compte : cloner `emph_book` (lecture emphatique de Jules Verne) avait donné
une voix jugée « très robot ». Le résolveur préfère `neut` et transcrit lui-même le clip,
car aucun clip neutre de SIWIS n'a de transcript fourni.

In [ ]:
!{sys.executable} -m pip install -q faster-whisper soundfile huggingface_hub
sys.path.insert(0, REPO + '/python')
import logging, shutil, json
logging.basicConfig(level=logging.INFO, format='%(message)s')

from lfm2_audio.data_prep.siwis_reference import resolve_reference
ref = resolve_reference()
shutil.copy(ref.wav_path, '/content/siwis_ref.wav')
open('/content/siwis_ref.txt', 'w').write(ref.text)
print('référence :', ref.stem, '|', ref.text[:80])

from IPython.display import Audio, display
display(Audio('/content/siwis_ref.wav'))

## 3. Pile vLLM épinglée

Recette du notebook `colab_vllm_omni_integration` : `vllm-omni` ne déclare pas `vllm`,
les versions sont appariées en major.minor, et le build PyPI de vllm 0.22 est CUDA 13
(absent de Colab) → wheel officiel `+cu129` + torch assorti.

Compter 6-8 min. **Si torch est réaligné, redémarrer la session puis reprendre ici** —
la référence SIWIS est déjà sur disque, elle survit au redémarrage.

In [ ]:
# Regarder AVANT d'installer. L'image Colab d'août 2026 livre déjà vllm 0.26.0 et
# vllm-omni 0.26.0 appariés, avec torch/torchaudio 2.11 cohérents. Installer la
# recette 0.22 par-dessus DOWNGRADE un environnement sain : elle tire un autre torch,
# ce qui casse torchaudio, ce qui casse la résolution de la référence SIWIS.
# Chaque panne de cette soirée venait du correctif précédent.
import importlib.metadata as md, sys

def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

v_vllm, v_omni = version('vllm'), version('vllm-omni')
print('déjà installé — vllm', v_vllm, '| vllm-omni', v_omni,
      '| torch', version('torch'), '| torchaudio', version('torchaudio'))

paired = v_vllm and v_omni and v_vllm.split('.')[:2] == v_omni.split('.')[:2]
if paired:
    print('✅ paire cohérente : on n installe RIEN')
else:
    # Repli : la recette vérifiée, pour une image qui ne les aurait pas.
    WHL = 'https://github.com/vllm-project/vllm/releases/download/v0.22.1/vllm-0.22.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl'
    !{sys.executable} -m pip install -q "vllm @ {WHL}" --extra-index-url https://download.pytorch.org/whl/cu129
    !{sys.executable} -m pip install -q "vllm-omni==0.22.0"
    print('⚠️ installé — Exécution → Redémarrer la session avant de continuer.')

!{sys.executable} -m pip install -q mistral-common


## 4. Libs CUDA 13 — à exécuter AVANT tout import de `vllm_omni`

Le wheel est buildé CUDA 13, le torch de Colab est cu12x : `import vllm_omni` échoue sur
`libcudart.so.13`. On précharge les `.so` de `nvidia/cu13` en `RTLD_GLOBAL` (ce kernel)
**et** on exporte `LD_LIBRARY_PATH` (hérité par les sous-process que l'engine spawn).

⚠️ Chercher **`nvidia/cu13/lib` exactement** — mon job avait élargi à `nvidia/*/lib` et
chargeait `cublas` en annonçant « cu13 préchargées ».

In [ ]:
# Le préchargement cu13 n'est nécessaire QUE si vllm vient du build PyPI (CUDA 13).
# Avec le wheel +cu129 les libs sont cu12x et fournies par torch : on TESTE l'import
# d'abord, on ne précharge que s'il échoue. L'assertion précédente supposait une
# configuration qui n'est pas celle-ci.
import glob, ctypes, site, os, traceback

try:
    import vllm, vllm_omni
    print('imports OK sans préchargement —', vllm.__version__, '|', vllm_omni.__version__)
except Exception:
    traceback.print_exc()
    roots = [*site.getsitepackages(), site.getusersitepackages()]
    cands = [d for r in roots for d in glob.glob(r + '/nvidia/cu13/lib')]
    print('candidats cu13 :', cands or 'AUCUN')
    for cu13 in cands:
        os.environ['LD_LIBRARY_PATH'] = cu13 + ':' + os.environ.get('LD_LIBRARY_PATH', '')
        for so in sorted(glob.glob(cu13 + '/lib*.so*')):
            try:
                ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass
    if cands:
        print('libs préchargées — RELANCER cette cellule')
    else:
        print("pas de cu13 : si l'erreur ci-dessus mentionne libcudart.so.13,\n"
              "installer 'nvidia-cuda-runtime-cu13' ; sinon la cause est ailleurs.")


## 5. Le point qui bloque

C'est ici que mon dernier run est mort :

```
from vllm import SamplingParams
  File "vllm/__init__.py", line 70, in __getattr__
    module = import_module(module_name, __package__)
AssertionError
```

Un `AssertionError` sans message, levé par l'import paresseux de `vllm/__init__.py`.
La cellule suivante isole l'import pour voir la vraie trace.

In [ ]:
import traceback
try:
    from vllm import SamplingParams
    print('SamplingParams OK')
except Exception:
    traceback.print_exc()
    # Import direct du sous-module pour contourner le __getattr__ paresseux :
    from vllm.sampling_params import SamplingParams
    print('contourné via vllm.sampling_params')

from vllm_omni.entrypoints.omni import Omni
print('Omni importé')

## 6. Synthèse — dix phrases, registre conversationnel

Ce sont les phrases que dit vraiment un assistant d'accueil, pas de la prose : une voix
qui brille sur du texte lu peut décevoir sur « Il est quinze heures trente ».

In [ ]:
from mistral_common.protocol.speech.request import SpeechRequest
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
import soundfile as sf

MODEL = 'mistralai/Voxtral-4B-TTS-2603'
SENTENCES = [
    "Bonjour, comment puis-je vous aider aujourd'hui ?",
    'Il est quinze heures trente, votre rendez-vous est dans une demi-heure.',
    "Je n'ai pas trouvé ce nom dans l'annuaire. Pouvez-vous me l'épeler ?",
    "D'accord, je préviens votre interlocutrice tout de suite.",
    'Le code du wifi invité est affiché sur le panneau derrière vous.',
    "Attendez, je vérifie... Oui, c'est bien confirmé pour jeudi.",
    "Désolé, je n'ai pas bien entendu. Vous pouvez répéter ?",
    'Avec plaisir ! Bonne journée et à bientôt.',
    'Alors, il y a deux possibilités : soit vous patientez, soit je vous rappelle.',
    'Je vous mets en relation, ne quittez pas.',
]

ref_audio = open('/content/siwis_ref.wav', 'rb').read()
tok = MistralTokenizer.from_hf_hub(MODEL).instruct_tokenizer
inputs = [{'prompt_token_ids': tok.encode_speech_request(
              SpeechRequest(input=s, ref_audio=ref_audio)).tokens} for s in SENTENCES]

engine = Omni(model=MODEL)
outputs = engine.generate(inputs, [SamplingParams(max_tokens=4096)] * len(inputs))

os.makedirs('/content/out/voxtral_tts', exist_ok=True)
for i, o in enumerate(outputs):
    audio = o.multimodal_output['audio'].tolist()
    sf.write(f'/content/out/voxtral_tts/s{i:02d}.wav', audio, 24000)
    open(f'/content/out/voxtral_tts/s{i:02d}.txt', 'w').write(SENTENCES[i])
    print(f's{i:02d} {len(audio)/24000:.1f}s')

## 7. Écoute

C'est le juge qui compte : votre oreille a détecté l'accent anglais de `qwen_preset` et
le registre théâtral du premier clone, deux choses qu'UTMOS notait bien.

In [ ]:
from IPython.display import Audio, display
for i in range(len(SENTENCES)):
    print(SENTENCES[i])
    display(Audio(f'/content/out/voxtral_tts/s{i:02d}.wav'))

## 8. Récupération

Colab n'a pas d'ingress : les fichiers meurent avec la VM. Poussez-les sur le Hub, ou
téléchargez l'archive avant de fermer.

In [ ]:
!cd /content/out && tar czf /content/voxtral_tts.tgz voxtral_tts
from google.colab import files
files.download('/content/voxtral_tts.tgz')